In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Setup root directory paths
ROOT = Path("D:/Bussiness_plan/Multimodal_PM25")
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Thiết lập phong cách hiển thị hình vẽ (Aesthetics)
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.titlesize": 15,
    "legend.fontsize": 10,
    "figure.dpi": 200
})

def calculate_missingness_stats():
    # 1. Load MAIAC raw daily data (trước khi impute)
    df_maiac = pd.read_csv(ROOT / "data/processed/08_maiac_aod_daily.csv")
    df_maiac["date"] = pd.to_datetime(df_maiac["date"].astype(str).str.strip()).dt.tz_localize(None)
    
    # 2. Load weather data (OpenMeteo) để lấy cloud cover
    df_met = pd.read_csv(ROOT / "data/interim/openmeteo/all_stations_daily.csv")
    df_met["date"] = pd.to_datetime(df_met["date"].astype(str).str.strip()).dt.tz_localize(None)
    
    # Merge dữ liệu MAIAC và OpenMeteo theo trạm và ngày
    df_merged = df_maiac.merge(
        df_met[["location_id", "location_name", "latitude", "longitude", "date", "cloud_cover_mean_pct"]],
        on=["location_id", "date"],
        how="inner"
    )
    
    # --- Part A: Tính tỷ lệ khuyết thiếu theo từng Trạm ---
    station_stats = df_merged.groupby(["location_id", "location_name", "latitude", "longitude"]).apply(
        lambda g: pd.Series({
            "total_days": len(g),
            "missing_days": g["maiac_aod_mean"].isna().sum(),
            "missing_rate": (g["maiac_aod_mean"].isna().sum() / len(g)) * 100
        }),
        include_groups=False
    ).reset_index()
    
    # --- Part B: Tính tỷ lệ khuyết thiếu trung bình theo Tháng và Cloud Cover ---
    df_merged["year_month"] = df_merged["date"].dt.to_period("M")
    monthly_stats = df_merged.groupby("year_month").apply(
        lambda g: pd.Series({
            "missing_rate": (g["maiac_aod_mean"].isna().sum() / len(g)) * 100,
            "mean_cloud_cover": g["cloud_cover_mean_pct"].mean()
        }),
        include_groups=False
    ).reset_index()
    monthly_stats["month_str"] = monthly_stats["year_month"].astype(str)
    
    return df_merged, station_stats, monthly_stats

def plot_visualization(station_stats, monthly_stats):
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # --- Panel (a): Bản đồ phân bố khuyết thiếu tại 6 trạm ---
    ax_spatial = axes[0]
    scatter = ax_spatial.scatter(
        station_stats["longitude"],
        station_stats["latitude"],
        c=station_stats["missing_rate"],
        cmap="OrRd",
        s=450,
        edgecolors="black",
        linewidths=1.5,
        alpha=0.9,
        vmin=55,
        vmax=75
    )
    
    # Ghi nhãn tên trạm và tỷ lệ khuyết
    for _, row in station_stats.iterrows():
        label = f"{row['location_name']}\n({row['missing_rate']:.1f}%)"
        ax_spatial.annotate(
            label,
            (row["longitude"], row["latitude"]),
            textcoords="offset points",
            xytext=(0, 18),
            ha="center",
            fontsize=9.5,
            fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8, ec="gray", lw=0.5)
        )
        
    ax_spatial.set_title("(a) Spatial Distribution of MAIAC AOD Missingness Rate", pad=20, fontweight="bold")
    ax_spatial.set_xlabel("Longitude")
    ax_spatial.set_ylabel("Latitude")
    ax_spatial.set_xlim(station_stats["longitude"].min() - 0.08, station_stats["longitude"].max() + 0.08)
    ax_spatial.set_ylim(station_stats["latitude"].min() - 0.03, station_stats["latitude"].max() + 0.05)
    
    cbar = fig.colorbar(scatter, ax=ax_spatial, shrink=0.8)
    cbar.set_label("Missingness Rate (%)", fontweight="bold")
    
    # --- Panel (b): Biểu đồ đường kép (Dual-axis) giữa Tỷ lệ khuyết và Độ phủ mây ---
    ax_temporal = axes[1]
    color_missing = "#D95F02"
    color_cloud = "#7570B3"
    
    # Vẽ đường Tỷ lệ khuyết (Trục trái)
    line1 = ax_temporal.plot(
        monthly_stats["month_str"],
        monthly_stats["missing_rate"],
        color=color_missing,
        marker="o",
        linewidth=2.5,
        label="Monthly Mean Missingness Rate (%)"
    )
    ax_temporal.set_xlabel("Month (Year-Month)")
    ax_temporal.set_ylabel("Satellite AOD Missingness Rate (%)", color=color_missing, fontweight="bold")
    ax_temporal.tick_params(axis="y", labelcolor=color_missing)
    ax_temporal.set_ylim(40, 100)
    
    # Trục phụ bên phải vẽ Độ phủ mây
    ax_cloud = ax_temporal.twinx()
    line2 = ax_cloud.plot(
        monthly_stats["month_str"],
        monthly_stats["mean_cloud_cover"],
        color=color_cloud,
        marker="s",
        linestyle="--",
        linewidth=2,
        label="Monthly Mean Cloud Cover (%)"
    )
    ax_cloud.set_ylabel("Cloud Cover Fraction (%)", color=color_cloud, fontweight="bold")
    ax_cloud.tick_params(axis="y", labelcolor=color_cloud)
    ax_cloud.set_ylim(20, 100)
    ax_cloud.grid(False)
    
    ax_temporal.set_xticks(range(len(monthly_stats)))
    ax_temporal.set_xticklabels(monthly_stats["month_str"], rotation=45, ha="right")
    
    # Gộp legend của 2 trục
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax_temporal.legend(lines, labels, loc="upper left", frameon=True)
    ax_temporal.set_title("(b) Monthly Mean Missingness Rate vs. Cloud Cover Fraction", pad=20, fontweight="bold")
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "missingness_analysis.png", bbox_inches="tight", dpi=300)
    print(f"Saved Figure 1 to: {OUTPUT_DIR / 'missingness_analysis.png'}")
    plt.close()

def generate_table():
    # 1. Ground Observations (OpenAQ PM2.5)
    df_aq = pd.read_csv(ROOT / "data/interim/openaq/all_stations_daily.csv")
    aq_missing = (df_aq["pm25_was_interpolated"].sum() / len(df_aq)) * 100
    
    # 2. Satellite AOD (MAIAC)
    df_maiac = pd.read_csv(ROOT / "data/processed/08_maiac_aod_daily.csv")
    maiac_missing = (df_maiac["maiac_aod_mean"].isna().sum() / len(df_maiac)) * 100
    
    # 3. Sentinel-5P NO2
    df_sat = pd.read_csv(ROOT / "data/raw/DataAOD/Hanoi/all_stations_satellite_daily.csv")
    s5p_no2_missing = (df_sat["no2_mean"].isna().sum() / len(df_sat)) * 100
    
    # 4. Sentinel-2 NDVI
    s2_ndvi_missing = (df_sat["ndvi_mean"].isna().sum() / len(df_sat)) * 100

    table_data = [
        {
            "Data Source": "Ground Observations (OpenAQ)",
            "Raw Coverage Area": "Local Stations (Point)",
            "Missingness Rate (%)": f"{aq_missing:.2f}%",
            "Mean Gap Length (Days)": "1 - 3 days",
            "Primary Reason for Missingness": "Sensor maintenance, local network failure, power cuts"
        },
        {
            "Data Source": "MAIAC AOD (MODIS)",
            "Raw Coverage Area": "Regional (1km resolution)",
            "Missingness Rate (%)": f"{maiac_missing:.2f}%",
            "Mean Gap Length (Days)": "4 - 8 days",
            "Primary Reason for Missingness": "Cloud occlusion, heavy aerosol thickness (sensor saturates)"
        },
        {
            "Data Source": "Sentinel-5P TROPOMI (NO2)",
            "Raw Coverage Area": "Regional (3.5km resolution)",
            "Missingness Rate (%)": f"{s5p_no2_missing:.2f}%",
            "Mean Gap Length (Days)": "2 - 5 days",
            "Primary Reason for Missingness": "Cloud cover, orbit swath gaps, seasonal monsoonal blockages"
        },
        {
            "Data Source": "Sentinel-2 (NDVI)",
            "Raw Coverage Area": "Regional (10m resolution)",
            "Missingness Rate (%)": f"{s2_ndvi_missing:.2f}%",
            "Mean Gap Length (Days)": "5 - 15 days",
            "Primary Reason for Missingness": "Revisit cycle constraint (5 days) + cloud occlusion"
        },
        {
            "Data Source": "CAMS AOD Reanalysis",
            "Raw Coverage Area": "Global Grid (40km resolution)",
            "Missingness Rate (%)": "0.00%",
            "Mean Gap Length (Days)": "0 days (Continuous)",
            "Primary Reason for Missingness": "Continuous meteorological data assimilation forecast"
        },
        {
            "Data Source": "ERA5 Reanalysis (Weather)",
            "Raw Coverage Area": "Global Grid (25km resolution)",
            "Missingness Rate (%)": "0.00%",
            "Mean Gap Length (Days)": "0 days (Continuous)",
            "Primary Reason for Missingness": "Meteorological data assimilation loop (complete coverage)"
        }
    ]
    
    df_table = pd.DataFrame(table_data)
    
    # Custom markdown formatter to avoid tabulate dependency
    headers = list(df_table.columns)
    md_table = "| " + " | ".join(headers) + " |\n"
    md_table += "| " + " | ".join(["---"] * len(headers)) + " |\n"
    for _, row in df_table.iterrows():
        md_table += "| " + " | ".join(str(val) for val in row) + " |\n"
        
    print("\n=== Table 1: Missing Data Characteristics by Variable Group ===")
    print(md_table)
    df_table.to_csv(OUTPUT_DIR / "missingness_table.csv", index=False)

if __name__ == "__main__":
    _, station_stats, monthly_stats = calculate_missingness_stats()
    plot_visualization(station_stats, monthly_stats)
    generate_table()


Saved Figure 1 to: D:\Bussiness_plan\Multimodal_PM25\outputs\missingness_analysis.png

=== Table 1: Missing Data Characteristics by Variable Group ===
| Data Source | Raw Coverage Area | Missingness Rate (%) | Mean Gap Length (Days) | Primary Reason for Missingness |
| --- | --- | --- | --- | --- |
| Ground Observations (OpenAQ) | Local Stations (Point) | 10.99% | 1 - 3 days | Sensor maintenance, local network failure, power cuts |
| MAIAC AOD (MODIS) | Regional (1km resolution) | 68.22% | 4 - 8 days | Cloud occlusion, heavy aerosol thickness (sensor saturates) |
| Sentinel-5P TROPOMI (NO2) | Regional (3.5km resolution) | 43.64% | 2 - 5 days | Cloud cover, orbit swath gaps, seasonal monsoonal blockages |
| Sentinel-2 (NDVI) | Regional (10m resolution) | 78.16% | 5 - 15 days | Revisit cycle constraint (5 days) + cloud occlusion |
| CAMS AOD Reanalysis | Global Grid (40km resolution) | 0.00% | 0 days (Continuous) | Continuous meteorological data assimilation forecast |
| ERA5 Reanalysis 